# Plant primary key defects
Creates copies of the dev tables with known duplicate and null key defects, and registers them, so validation runs show how failures are reported.

In [0]:
%sql
-- customer with 5 duplicated rows: expect duplicate_count = 5
create or replace table validation_sweep.dev.customer_dupes as
select * from validation_sweep.dev.customer
union all
select * from validation_sweep.dev.customer where c_custkey <= 5;

-- partsupp with 4 null key values: expect null_key_count = 4
create or replace table validation_sweep.dev.partsupp_nulls as
select * from validation_sweep.dev.partsupp
union all
select ps_partkey, null, ps_availqty, ps_supplycost, ps_comment
from validation_sweep.dev.partsupp
where ps_partkey = 1;

In [0]:
%sql
-- Register the defect tables; delete first so reruns don't add duplicate entries
delete from validation_sweep.metadata.pk_registry
where schema_name = 'dev' and table_name in ('customer_dupes', 'partsupp_nulls');

insert into validation_sweep.metadata.pk_registry values
  ('validation_sweep', 'dev', 'customer_dupes', array('c_custkey'),                true, current_timestamp()),
  ('validation_sweep', 'dev', 'partsupp_nulls', array('ps_partkey', 'ps_suppkey'), true, current_timestamp());

In [0]:
# Verify the defect tables have the expected number of rows
expected_counts = {
    "validation_sweep.dev.customer_dupes": 150005,
    "validation_sweep.dev.partsupp_nulls": 800004,
}

errors = []
for table_name, expected in expected_counts.items():
    actual = spark.table(table_name).count()
    status = "OK" if actual == expected else "FAIL"
    print(f"{table_name:40} expected {expected:>9,}  found {actual:>9,}  {status}")
    if actual != expected:
        errors.append(f"{table_name}: expected {expected} rows, found {actual}")

if errors:
    raise ValueError("\n".join(errors))